# Семинар 7 - процессы

Сегодня в программе:
* `fork`
* `exec`
* `dup2`
* `pipe`
* Историческая справка


Неформально говоря, процесс - это экземпляр исполнения программы в операционной системе. 

Формально говоря, процесс - это экземпляр следующей структуры в ядре:
```c
struct task_struct {
/* these are hardcoded - don't touch */
  volatile long        state;          /* -1 unrunnable, 0 runnable, >0 stopped */
  long                 counter;
  long                 priority;
  unsigned             long signal;
  unsigned             long blocked;   /* bitmap of masked signals */
  unsigned             long flags;     /* per process flags, defined below */
  int errno;
  long                 debugreg[8];    /* Hardware debugging registers */
  struct exec_domain   *exec_domain;
/* various fields */
  struct linux_binfmt  *binfmt;
  struct task_struct   *next_task, *prev_task;
  struct task_struct   *next_run,  *prev_run;
  unsigned long        saved_kernel_stack;
  unsigned long        kernel_stack_page;
  int                  exit_code, exit_signal;
  /* ??? */
  unsigned long        personality;
  int                  dumpable:1;
  int                  did_exec:1;
  int                  pid;
  int                  pgrp;
  int                  tty_old_pgrp;
  int                  session;
  /* boolean value for session group leader */
  int                  leader;
  int                  groups[NGROUPS];
  /* 
   * pointers to (original) parent process, youngest child, younger sibling,
   * older sibling, respectively.  (p->father can be replaced with 
   * p->p_pptr->pid)
   */
  struct task_struct   *p_opptr, *p_pptr, *p_cptr, 
                       *p_ysptr, *p_osptr;
  struct wait_queue    *wait_chldexit;  
  unsigned short       uid,euid,suid,fsuid;
  unsigned short       gid,egid,sgid,fsgid;
  unsigned long        timeout, policy, rt_priority;
  unsigned long        it_real_value, it_prof_value, it_virt_value;
  unsigned long        it_real_incr, it_prof_incr, it_virt_incr;
  struct timer_list    real_timer;
  long                 utime, stime, cutime, cstime, start_time;
/* mm fault and swap info: this can arguably be seen as either
   mm-specific or thread-specific */
  unsigned long        min_flt, maj_flt, nswap, cmin_flt, cmaj_flt, cnswap;
  int swappable:1;
  unsigned long        swap_address;
  unsigned long        old_maj_flt;    /* old value of maj_flt */
  unsigned long        dec_flt;        /* page fault count of the last time */
  unsigned long        swap_cnt;       /* number of pages to swap on next pass */
/* limits */
  struct rlimit        rlim[RLIM_NLIMITS];
  unsigned short       used_math;
  char                 comm[16];
/* file system info */
  int                  link_count;
  struct tty_struct    *tty;           /* NULL if no tty */
/* ipc stuff */
  struct sem_undo      *semundo;
  struct sem_queue     *semsleeping;
/* ldt for this task - used by Wine.  If NULL, default_ldt is used */
  struct desc_struct *ldt;
/* tss for this task */
  struct thread_struct tss;
/* filesystem information */
  struct fs_struct     *fs;
/* open file information */
  struct files_struct  *files;
/* memory management info */
  struct mm_struct     *mm;
/* signal handlers */
  struct signal_struct *sig;
#ifdef __SMP__
  int                  processor;
  int                  last_processor;
  int                  lock_depth;     /* Lock depth. 
                                          We can context switch in and out
                                          of holding a syscall kernel lock... */  
#endif   
};
```

Процесс может находится в нескольких возможных состояниях:

* Выполняется
* Остановлен (до получения сигнала SIGCONT, о них поговорим позже)
* Сон (например, ждёт ввода)
* Зомби (процесс завершился, но ещё не удалён из таблицы процессов)

<img src="media/process_states.png" alt="process states" width="700" style="background-color:white;"/>

### Создание процессов. fork

Системный вызов `fork` создаёт почти полную копию текущего процесса за исключением:

* pid
* маски сигналов, ожидающих обработки
* блокировок памяти и файлов
* таймеров
* операция неблокирующего чтения
* ...

`pid_t fork(void)`

В родителе при успешном он возвращает `pid` ребёнка, а в самом ребёнке `0`. Чтобы получить `pid` в самом ребёнке можно, например, использовать системный вызов `getpid`.

Но раз вы создали некоторый ресурс, нужно не забыть его удалить (иначе в таблице процессов кончится место). Для этого служит семейство семейных вызовов `wait`. Наиболее часто для этого используют `waitpid`.

`pid_t waitpid(pid_t pid, int *wstatus, int options);`

Он позволяет дождаться процесса по `pid` и получить его статус возврата. С помощью `WEXITSTATUS(wstatus)` можно вытащить из него код возврата. По-умолчанию `waitpid` ждёт перехода в зомби, но можно с помощью опцию переопределить это поведение на какие-то другие состояния процесса.

In [5]:
!gcc snippets/simple_fork.c -o snippets/simple_fork.out 
!cat snippets/simple_fork.c

#include <stdio.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>

int main() {
    pid_t pid = fork();
    if (pid == 0) {
        printf("Hello from child with pid %d!\n", getpid());
        return 42;
    } else {
        int status;
        waitpid(pid, &status, 0);
        printf("Child exited with code %d\n", WEXITSTATUS(status));
        printf("Hello from parent!\n");
    }
}


In [10]:
!./snippets/simple_fork.out

Hello from child with pid 87059!
Child exited with code 42
Hello from parent!


#### С помощью флага `MAP_SHARED` можно передать от ребёнка какую-то информацию родителю:

In [11]:
!gcc snippets/mmap_shared.c -o snippets/mmap_shared.out 
!cat snippets/mmap_shared.c

#include <stdio.h>
#include <sys/mman.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>
#include <string.h>

int main() {
    char* content_ptr = mmap(NULL, 10, PROT_READ | PROT_WRITE, MAP_SHARED | MAP_ANONYMOUS, -1, 0);
    strcpy(content_ptr, "Hello");
    pid_t pid = fork();
    if (pid == 0) {
        printf("Parent says: %s!\n", content_ptr);
        strcpy(content_ptr, "Hola");
    } else {
        int status;
        waitpid(pid, &status, 0);
        printf("Child says: %s!\n", content_ptr);
    }
    return 0;
}


In [12]:
!./snippets/mmap_shared.out

Parent says: Hello!
Child says: Hola!


При работе с памятью после `fork` используется подход Copy-on-Write. То есть при `fork` не происходит мгновенного копирования всей используемой процессом памяти. 

#### Также важно учитывать, что `fork` ничего не знает об инвариантах вашей программы, это может привести к странному поведению:

In [13]:
!gcc snippets/fork-buffer.c -o snippets/fork-buffer.out 
!cat snippets/fork-buffer.c

#include <stdio.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <unistd.h>

int main() {
    printf("Hello, world!");
    pid_t pid = fork();
    if (pid == 0) {
        printf("Hello from child with pid %d!\n", getpid());
        return 0;
    } else {
        int status;
        waitpid(pid, &status, 0);
        printf("Hello from parent!\n");
    }
}


In [14]:
!./snippets/fork-buffer.out

Hello, world!Hello from child with pid 91358!
Hello, world!Hello from parent!


Вывод `printf`, сделанный еще до вызова `fork`, задублировался. Это произошло из-за того, что `printf` буфферизует вывод. 

#### С помощью `fork` можно сделать форк-бомбу:

In [16]:
!cat snippets/fork-bomb.c

#include <unistd.h>

int main() {
    while (1) {
        fork();
    }
}


Крайне не рекомендуется запускать её в неподготовленной для этого среде. 

### exec

Обычно системный вызов `fork` используется в комбинации с другим системным вызовом `exec`, который запускает программу. Виртуальное адресное пространство при этом перезаписывается. Большинство других атрибутов процесса, например, таблица файловых дескрипторов, сохраняются. Подробнее можно почитать в `man 2 execve`.

Поверх этого системногого вызова реализованы в библиотеке разные интерфейсы:

```c
int execve(const char *filename,
           char *const argv[],
           char *const envp[]);           
int execvpe(.....) // параметры аналогично execve

int execv(const char *filename, char *const argv[])
int execvp(......) // параметры аналогично execv

int execle(const char *filename,
           const char arg0, ..., /* NULL */,
           const char env0, ..., /* NULL */);

int execl(const char *filename,
          const char arg0, ..., /* NULL */);
int execlp(......) // параметры аналогично execl
```

Что означают суффиксы?

* `v` или `l` - параметры передаются в виде массивов (`v`), заканчивающихся элементом `NULL`, либо в виде переменного количества аргументов (`l`), где признаком конца перечисления аргументов является значение `NULL`
* `e` - кроме аргументов программы передаются переменные окружения в виде строк `КЛЮЧ=ЗНАЧЕНИЕ`.
* `p` - именем программы может быть не только имя файла, но и имя, которое нужно найти в одном из каталогов, перечисленных в переменной окружения `PATH`.

Вооружившись `fork+exec` можно, например, написать свой простенький `shell`:

In [18]:
!gcc snippets/simple_shell.c -o snippets/simple_shell.out 
!cat snippets/simple_shell.c

#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <sys/types.h>
#include <sys/wait.h>
#include <string.h>

int main() {
    size_t BUFF_SIZE = 4096;
    char* shell_intro = "my_shell> ";
    printf("%s", shell_intro);

    char* buff = malloc(BUFF_SIZE);
    size_t read;
    while((read = getline(&buff, &BUFF_SIZE, stdin)) != -1){
        size_t MAX_ARGS = 10;
        char* args[MAX_ARGS];
        for (int i = 0; i < MAX_ARGS; i++) {
            args[i] = NULL;
        }
        char* pos;
        pos = strtok(buff, " \n");
        int argc = 0;
        while (pos != NULL) {
            args[argc] = pos;
            ++argc;
            pos = strtok(NULL, " \n");
        }

        pid_t pid = fork();
        if (pid == 0) {
            execvp(args[0], args);
            perror("Can't spawn child: ");
        } else {
            waitpid(pid, NULL, 0);
            printf("%s", shell_intro);
        }
    }

    return 0;
}


В настоящем shell есть еще перенаправление ввода/вывода (`>`, `|`), как такое сделать? 

### dup2

Сисколы `dup` и `dup2` выполняют дубликацию файловых дескрипторов:

```c
#include <unistd.h>

int dup(int fd);  // duplicate fd, return new descriptor with the same meaning
int dup2(int fd, int fd2);  // same as dup, but new descriptor value can be specified
```

Простейший пример использования: 

In [19]:
!gcc snippets/simple_dup2.c -o snippets/simple_dup2.out 
!cat snippets/simple_dup2.c

#include <stdio.h>
#include <unistd.h>
#include <assert.h>
#include <fcntl.h>
#include <sys/types.h>


int main() {
    int fd = open("out.txt", O_WRONLY | O_CREAT | O_TRUNC, 0664);
    dup2(fd, 1); // redirect stdout to file
    close(fd);
    printf("Redirectred 'Hello world!'");
    return 0;
}

In [20]:
!./snippets/simple_dup2.out

Более продвинутый пример использования в сочетании с `exec`: делаем программу, запускающую другую программу и перенаправляющую ее вывод в файл:

In [21]:
!gcc snippets/output_redirect.c -o snippets/output_redirect.out 
!cat snippets/output_redirect.c

#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <assert.h>
#include <fcntl.h>
#include <sys/resource.h>
#include <sys/types.h>
#include <sys/wait.h>


int main(int argc, char** argv) {
    assert(argc >= 2);
    int fd = open(argv[1], O_WRONLY | O_CREAT | O_TRUNC, 0664);
    assert(fd >= 0);
    dup2(fd, STDOUT_FILENO);
    close(fd);
    execvp(argv[2], argv + 2);
    assert(0 && "Unreachable position in code if execlp succeeded");
}

In [27]:
!./snippets/output_redirect.out out.txt echo 'Hello, world' 

Теперь реализауем перенаправление вывода одной программы на ввод другой (`|`). Для этого нам потребуется еще один сискол: `pipe`:

```c
#include <unistd.h>

int pipe(int fd[2]);
```

Он создает пайп - то есть пару связанных файловых дескрипторов, в один из которых можно писать, а из другого читать.

Теперь, используя `pipe`, мы можем реализовать `|` - пайп :) 

In [28]:
!gcc snippets/pipe.c -o snippets/pipe.out 
!cat snippets/pipe.c

#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <assert.h>
#include <fcntl.h>
#include <sys/resource.h>
#include <sys/types.h>
#include <sys/wait.h>


int main() {
    int fd[2];
    pipe(fd); // fd[0] - in, fd[1] - out (like stdin=0, stdout=1)
    pid_t pid_1, pid_2;
    if ((pid_1 = fork()) == 0) {
        dup2(fd[1], 1);
        close(fd[0]);
        close(fd[1]);
        execlp("ps", "ps", "aux", NULL);
        assert(0 && "Unreachable position in code if execlp succeeded");
    }
    close(fd[1]);
    
    if ((pid_2 = fork()) == 0) {
        dup2(fd[0], 0);
        close(fd[0]);
        execlp("tail", "tail", "-n", "4", NULL);
        assert(0 && "Unreachable position in code if execlp succeeded");
    }
    close(fd[0]);
    
    int status;
    assert(waitpid(pid_1, &status, 0) != -1);
    assert(waitpid(pid_2, &status, 0) != -1);
    return 0;
}

In [29]:
!./snippets/pipe.out

k.afentev        76303   0.0  0.1 426980048  39840   ??  Ss    8:32AM   0:00.24 /System/Library/Frameworks/Metal.framework/Versions/A/XPCServices/MTLCompilerService.xpc/Contents/MacOS/MTLCompilerService
k.afentev        76302   0.0  0.1 426980048  46032   ??  Ss    8:32AM   0:00.64 /System/Library/Frameworks/Metal.framework/Versions/A/XPCServices/MTLCompilerService.xpc/Contents/MacOS/MTLCompilerService
k.afentev        74892   0.0  0.1 411230768  26080   ??  Ss    8:29AM   0:00.07 /System/Library/Frameworks/VideoToolbox.framework/Versions/A/XPCServices/VTEncoderXPCService.xpc/Contents/MacOS/VTEncoderXPCService
k.afentev        74142   0.0  0.5 1865550464 192896   ??  S     8:28AM   0:01.03 /Applications/Google Chrome.app/Contents/Frameworks/Google Chrome Framework.framework/Versions/141.0.7390.123/Helpers/Google Chrome Helper (Renderer).app/Contents/MacOS/Google Chrome Helper (Renderer) --type=renderer --enable-dinosaur-easter-egg-alt-images --origin-trial-disabled-features=CanvasTextN

### Немного истории

`fork`, как и связка `fork + exec` были придуманы в 1970-х. Простота рассматривалась как одно из преимуществ этого подхода к созданию новых процессов. Но сейчас, по прошествии многих лет, считается, что `fork` имеет множество недостатков:
* `fork` больше не такой простой, как кажется. Сейчас его реализаций содержит множество оговорок насчет блокировок, таймеров, асинхронного ввода-вывода и т.д. 
* `fork` проектировался без учета возможноси многопоточного исполнения, сейчас это влечет за собой проблемы
* `fork` не соответсвтует современным соображениям безопасности: например, он наследует почти все аттрибуты и права родительского процесса
* `fork` непроизводительный. Это может проявляться в реальных сервисах и приложениях ([up to 100ms delays in Chromium due to fork](bugs.chromium.org/p/chromium/issues/detail?id=819228)) <img src="media/fork_vs_spawn.png" alt="fork vs spawn performance" width="500" style="background-color:white;"/>
* `fork` стимулирует over-commitment памяти

Сейчас есть несколько более приемлемых альтернатив, в зависимости от целей: `posix_spawn`, `vfork`, `clone`